### Regressors

We now train a Random Forest Regressor and a Gradient Boosting Regressor for each position to predict the number of points a player will score in the next game (assuming they play). Only players with 'minutes_over_60' = 1 are included. We use `optuna` to find the best values for model hyperparameters using Bayesian Optimisation.

In [3]:
import pandas as pd

# add 'over_60_minutes' column
df = pd.read_csv(../data_preparation/previous_seasons_dataset.csv')
df['over_60_minutes'] = (df['minutes'] >= 60).astype(int)

In [4]:
import optuna
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import logging

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial, model_type, position):
    
    
    data = df[(df['position'] == position) & (df['over_60_minutes'] == 1)]
        
    x = data[['team_market_value', 'opponent_market_value', 'value', 'was_home','points_last_game', 'total_points', 'mins_last_game',
                        'total_mins', 'mean_points_last_3', 'mean_mins_last_3', 'mean_points_last_5','mean_mins_last_5', 'mean_points_last_10', 
                        'mean_mins_last_10', 'team_points_last_game', 'total_team_points', 'mean_team_points_last_3', 'mean_team_points_last_5',
                        'mean_team_points_last_10', 'team_conceded_last_game', 'total_team_conceded', 
                        'mean_team_conceded_last_3', 'mean_team_conceded_last_5', 'mean_team_conceded_last_10', 'total_opponent_points',
                        'opponent_points_last_game', 'mean_opponent_points_last_3', 'mean_opponent_points_last_5', 'mean_opponent_points_last_10',
                        'total_opponent_conceded', 'opponent_conceded_last_game', 'mean_opponent_conceded_last_3', 'mean_opponent_conceded_last_5',
                        'mean_opponent_conceded_last_10', 'total_points_last_season', 'total_mins_last_season', 'total_team_points_last_season',
                        'total_team_conceded_last_season', 'total_opponent_points_last_season', 'total_opponent_conceded_last_season']] 
    y = data['points']

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
    
    # Define hyperparameters for RandomForest and GradientBoosting
    if model_type == 'random_forest':
        n_estimators = trial.suggest_int('n_estimators', 50, 1000)
        max_depth = trial.suggest_int('max_depth', 3, 50)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
        max_features = trial.suggest_categorical('max_features', [None, 'sqrt', 'log2'])
        
        model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
                                      max_features=max_features, random_state=42)
    
    elif model_type == 'gradient_boosting':
        n_estimators = trial.suggest_int('n_estimators', 50, 1000)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.5, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 10)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
        subsample = trial.suggest_float('subsample', 0.5, 1.0)
        max_features = trial.suggest_categorical('max_features', [None, 'sqrt', 'log2'])

        model = GradientBoostingRegressor(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, min_samples_split=min_samples_split,
                                          min_samples_leaf=min_samples_leaf, subsample=subsample, max_features=max_features, random_state=42)

    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    return rmse

# Function to run optimization for both models across positions
def run_optimization(df, positions):
    best_params = {}
    for position in positions:
        # RandomForest Optimization
        study_rf = optuna.create_study(direction='minimize')
        study_rf.optimize(lambda trial: objective(trial, 'random_forest', position), n_trials=50)
        best_params[(position, 'random_forest')] = study_rf.best_params
        print(f"Best Random Forest params for {position}: {study_rf.best_params}, best RMSE: {study_rf.best_value}")

        # GradientBoosting Optimization
        study_gb = optuna.create_study(direction='minimize')
        study_gb.optimize(lambda trial: objective(trial, 'gradient_boosting', position), n_trials=50)
        best_params[(position, 'gradient_boosting')] = study_gb.best_params
        print(f"Best Gradient Boosting params for {position}: {study_gb.best_params}, RMSE: {study_gb.best_value}")
        
    return best_params

positions = ['GK', 'DEF', 'MID', 'FWD']
best_hyperparameters = run_optimization(df, positions)

Best Random Forest params for GK: {'n_estimators': 595, 'max_depth': 3, 'min_samples_split': 15, 'min_samples_leaf': 18, 'max_features': 'sqrt'}, best RMSE: 2.80943352291735
Best Gradient Boosting params for GK: {'n_estimators': 194, 'learning_rate': 0.013483111853881102, 'max_depth': 3, 'min_samples_split': 19, 'min_samples_leaf': 11, 'subsample': 0.9597819427689183, 'max_features': 'sqrt'}, RMSE: 2.823358461883462
Best Random Forest params for DEF: {'n_estimators': 461, 'max_depth': 32, 'min_samples_split': 9, 'min_samples_leaf': 2, 'max_features': None}, best RMSE: 2.2194397566079846
Best Gradient Boosting params for DEF: {'n_estimators': 515, 'learning_rate': 0.020413971141338902, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 14, 'subsample': 0.9021698488900685, 'max_features': None}, RMSE: 2.2668386113232217
Best Random Forest params for MID: {'n_estimators': 172, 'max_depth': 34, 'min_samples_split': 6, 'min_samples_leaf': 8, 'max_features': 'sqrt'}, best RMSE: 3.0

Using the best values for the hyperparameters we then train the models for each position.

In [5]:
GK_data = df[(df['position'] == 'GK') & (df['over_60_minutes'] == 1)]
GK_points_target = GK_data['points']
GK_points_features = GK_data[['team_market_value', 'opponent_market_value', 'value', 'was_home','points_last_game', 'total_points', 'mins_last_game',
                        'total_mins', 'mean_points_last_3', 'mean_mins_last_3', 'mean_points_last_5','mean_mins_last_5', 'mean_points_last_10', 
                        'mean_mins_last_10', 'team_points_last_game', 'total_team_points', 'mean_team_points_last_3', 'mean_team_points_last_5',
                        'mean_team_points_last_10', 'team_conceded_last_game', 'total_team_conceded', 
                        'mean_team_conceded_last_3', 'mean_team_conceded_last_5', 'mean_team_conceded_last_10', 'total_opponent_points',
                        'opponent_points_last_game', 'mean_opponent_points_last_3', 'mean_opponent_points_last_5', 'mean_opponent_points_last_10',
                        'total_opponent_conceded', 'opponent_conceded_last_game', 'mean_opponent_conceded_last_3', 'mean_opponent_conceded_last_5',
                        'mean_opponent_conceded_last_10', 'total_points_last_season', 'total_mins_last_season', 'total_team_points_last_season',
                        'total_team_conceded_last_season', 'total_opponent_points_last_season', 'total_opponent_conceded_last_season']]

x_train, x_test, y_train, y_test = train_test_split(GK_points_features, GK_points_target, train_size=0.8, test_size=0.2)

best_GK_rf_params = best_hyperparameters[('GK', 'random_forest')]

GK_rf_reg = RandomForestRegressor(n_estimators=best_GK_rf_params['n_estimators'], min_samples_split=best_GK_rf_params['min_samples_split'],
                                max_depth=best_GK_rf_params['max_depth'], min_samples_leaf=best_GK_rf_params['min_samples_leaf'],
                                max_features=best_GK_rf_params['max_features'], random_state=42)

cv_scores = cross_val_score(GK_rf_reg, x_train, y_train, cv=5, scoring= 'neg_root_mean_squared_error', n_jobs=-1)
GK_rf_reg.fit(x_train, y_train)
y_pred = GK_rf_reg.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = GK_rf_reg.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': GK_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))

print(f'Random Forest cross validation scores: {np.abs(cv_scores)}')
print(f'Random Forest mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'Random Forest RMSE: {RMSE}, Random Forest R-squared: {R_squared}')
print('-'*100)

best_GK_gb_params = best_hyperparameters[('GK', 'gradient_boosting')]

GK_gb_reg = GradientBoostingRegressor(n_estimators=best_GK_gb_params['n_estimators'], learning_rate=best_GK_gb_params['learning_rate'],
                                    max_depth=best_GK_gb_params['max_depth'], max_features=best_GK_gb_params['max_features'],
                                    min_samples_leaf=best_GK_gb_params['min_samples_leaf'], min_samples_split=best_GK_gb_params['min_samples_split'],
                                    subsample=best_GK_gb_params['subsample'] )

cv_scores = cross_val_score(GK_gb_reg, x_train, y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
GK_gb_reg.fit(x_train, y_train)
y_pred = GK_gb_reg.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = GK_gb_reg.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': GK_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))

print(f'Gradient Boosting cross validation scores: {np.abs(cv_scores)}')
print(f'Gradient Boosting mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'Gradient Boosting RMSE: {RMSE}, Gradient Boosting R-squared: {R_squared}')

                              Feature  Importance
1               opponent_market_value    0.093344
38  total_opponent_points_last_season    0.071508
2                               value    0.065402
33     mean_opponent_conceded_last_10    0.043620
34           total_points_last_season    0.043396
0                   team_market_value    0.041901
28       mean_opponent_points_last_10    0.041352
17            mean_team_points_last_5    0.039127
37    total_team_conceded_last_season    0.038533
22          mean_team_conceded_last_5    0.037373
Random Forest cross validation scores: [2.78510172 2.9041568  2.64784756 2.98339242 2.74687359]
Random Forest mean cross validation score: 2.8134744182152502
Random Forest RMSE: 2.8633406125206284, Random Forest R-squared: 0.01111032707086046
----------------------------------------------------------------------------------------------------
                              Feature  Importance
1               opponent_market_value    0.089393
2     

Defenders:

In [6]:
DEF_data = df[(df['position'] == 'DEF') & (df['over_60_minutes'] == 1)]
DEF_points_target = DEF_data['points']
DEF_points_features = DEF_data[['team_market_value', 'opponent_market_value', 'value', 'was_home','points_last_game', 'total_points', 'mins_last_game',
                        'total_mins', 'mean_points_last_3', 'mean_mins_last_3', 'mean_points_last_5','mean_mins_last_5', 'mean_points_last_10', 
                        'mean_mins_last_10', 'team_points_last_game', 'total_team_points', 'mean_team_points_last_3', 'mean_team_points_last_5',
                        'mean_team_points_last_10','team_conceded_last_game', 'total_team_conceded', 
                        'mean_team_conceded_last_3', 'mean_team_conceded_last_5', 'mean_team_conceded_last_10', 'total_opponent_points',
                        'opponent_points_last_game', 'mean_opponent_points_last_3', 'mean_opponent_points_last_5', 'mean_opponent_points_last_10',
                        'total_opponent_conceded', 'opponent_conceded_last_game', 'mean_opponent_conceded_last_3', 'mean_opponent_conceded_last_5',
                        'mean_opponent_conceded_last_10', 'total_points_last_season', 'total_mins_last_season', 'total_team_points_last_season',
                        'total_team_conceded_last_season', 'total_opponent_points_last_season', 'total_opponent_conceded_last_season']]

x_train, x_test, y_train, y_test = train_test_split(DEF_points_features, DEF_points_target, train_size=0.8, test_size=0.2)

# Random Forest Regressor

best_DEF_rf_params = best_hyperparameters[('DEF', 'random_forest')]

DEF_rf_reg = RandomForestRegressor(n_estimators=best_DEF_rf_params['n_estimators'], min_samples_split=best_DEF_rf_params['min_samples_split'],
                                max_depth=best_DEF_rf_params['max_depth'], min_samples_leaf=best_DEF_rf_params['min_samples_leaf'],
                                max_features=best_DEF_rf_params['max_features'], random_state=42)

cv_scores = cross_val_score(GK_rf_reg, x_train, y_train, cv=5, scoring= 'neg_root_mean_squared_error', n_jobs=-1)
DEF_rf_reg.fit(x_train, y_train)
y_pred = DEF_rf_reg.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = DEF_rf_reg.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': DEF_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))

print(f'Random Forest cross validation scores: {np.abs(cv_scores)}')
print(f'Random Forest mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'Random Forest RMSE: {RMSE}, Random Forest R-squared: {R_squared}')
print('-'*100)

# Gradient Boosting Regressor
best_DEF_gb_params = best_hyperparameters[('DEF', 'gradient_boosting')]

DEF_gb_reg = GradientBoostingRegressor(n_estimators=best_DEF_gb_params['n_estimators'], learning_rate=best_DEF_gb_params['learning_rate'],
                                    max_depth=best_DEF_gb_params['max_depth'], max_features=best_DEF_gb_params['max_features'],
                                    min_samples_leaf=best_DEF_gb_params['min_samples_leaf'], min_samples_split=best_DEF_gb_params['min_samples_split'],
                                    subsample=best_DEF_gb_params['subsample'] )

cv_scores = cross_val_score(DEF_gb_reg, x_train, y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
DEF_gb_reg.fit(x_train, y_train)
y_pred = DEF_gb_reg.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = DEF_gb_reg.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': DEF_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))

print(f'Gradient Boosting cross validation scores: {np.abs(cv_scores)}')
print(f'Gradient Boosting mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'Gradient Boosting RMSE: {RMSE}, Gradient Boosting R-squared: {R_squared}')

                           Feature  Importance
1            opponent_market_value    0.073030
0                team_market_value    0.046198
36   total_team_points_last_season    0.035764
31   mean_opponent_conceded_last_3    0.032809
30     opponent_conceded_last_game    0.031849
17         mean_team_points_last_5    0.031490
23      mean_team_conceded_last_10    0.030676
18        mean_team_points_last_10    0.030581
16         mean_team_points_last_3    0.030547
33  mean_opponent_conceded_last_10    0.030133
Random Forest cross validation scores: [3.07135481 2.93910502 3.06517035 3.10397548 3.06826518]
Random Forest mean cross validation score: 3.0495741708864577
Random Forest RMSE: 2.3188613533070406, Random Forest R-squared: 0.42690589756644715
----------------------------------------------------------------------------------------------------
                          Feature  Importance
1           opponent_market_value    0.066795
0               team_market_value    0.046406
3

Midfielders:

In [7]:
MID_data = df[(df['position'] == 'MID') & (df['over_60_minutes'] == 1)]
MID_points_target = MID_data['points']
MID_points_features = MID_data[['team_market_value', 'opponent_market_value', 'value', 'was_home','points_last_game', 'total_points', 'mins_last_game',
                        'total_mins', 'mean_points_last_3', 'mean_mins_last_3', 'mean_points_last_5','mean_mins_last_5', 'mean_points_last_10', 
                        'mean_mins_last_10', 'team_points_last_game', 'total_team_points', 'mean_team_points_last_3', 'mean_team_points_last_5',
                        'mean_team_points_last_10', 'team_conceded_last_game', 'total_team_conceded', 
                        'mean_team_conceded_last_3', 'mean_team_conceded_last_5', 'mean_team_conceded_last_10', 'total_opponent_points',
                        'opponent_points_last_game', 'mean_opponent_points_last_3', 'mean_opponent_points_last_5', 'mean_opponent_points_last_10',
                        'total_opponent_conceded', 'opponent_conceded_last_game', 'mean_opponent_conceded_last_3', 'mean_opponent_conceded_last_5',
                        'mean_opponent_conceded_last_10', 'total_points_last_season', 'total_mins_last_season', 'total_team_points_last_season',
                        'total_team_conceded_last_season', 'total_opponent_points_last_season', 'total_opponent_conceded_last_season']]

x_train, x_test, y_train, y_test = train_test_split(MID_points_features, MID_points_target, train_size=0.8, test_size=0.2)

# Random Forest Regressor

best_MID_rf_params = best_hyperparameters[('MID', 'random_forest')]

MID_rf_reg = RandomForestRegressor(n_estimators=best_MID_rf_params['n_estimators'], min_samples_split=best_MID_rf_params['min_samples_split'],
                                max_depth=best_MID_rf_params['max_depth'], min_samples_leaf=best_MID_rf_params['min_samples_leaf'],
                                max_features=best_MID_rf_params['max_features'], random_state=42)

cv_scores = cross_val_score(GK_rf_reg, x_train, y_train, cv=5, scoring= 'neg_root_mean_squared_error', n_jobs=-1)
MID_rf_reg.fit(x_train, y_train)
y_pred = MID_rf_reg.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = MID_rf_reg.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': MID_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))

print(f'Random Forest cross validation scores: {np.abs(cv_scores)}')
print(f'Random Forest mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'Random Forest RMSE: {RMSE}, Random Forest R-squared: {R_squared}')
print('-'*100)

# Gradient Boosting Regressor
best_MID_gb_params = best_hyperparameters[('MID', 'gradient_boosting')]

MID_gb_reg = GradientBoostingRegressor(n_estimators=best_MID_gb_params['n_estimators'], learning_rate=best_MID_gb_params['learning_rate'],
                                    max_depth=best_MID_gb_params['max_depth'], max_features=best_MID_gb_params['max_features'],
                                    min_samples_leaf=best_MID_gb_params['min_samples_leaf'], min_samples_split=best_MID_gb_params['min_samples_split'],
                                    subsample=best_MID_gb_params['subsample'] )

cv_scores = cross_val_score(MID_gb_reg, x_train, y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
MID_gb_reg.fit(x_train, y_train)
y_pred = MID_gb_reg.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = MID_gb_reg.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': MID_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))

print(f'Gradient Boosting cross validation scores: {np.abs(cv_scores)}')
print(f'Gradient Boosting mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'Gradient Boosting RMSE: {RMSE}, Gradient Boosting R-squared: {R_squared}')

                           Feature  Importance
2                            value    0.083112
34        total_points_last_season    0.061517
12             mean_points_last_10    0.038584
5                     total_points    0.032709
1            opponent_market_value    0.031901
10              mean_points_last_5    0.030045
0                team_market_value    0.027869
33  mean_opponent_conceded_last_10    0.027431
35          total_mins_last_season    0.026246
23      mean_team_conceded_last_10    0.025056
Random Forest cross validation scores: [2.94109595 3.08908251 3.05921264 2.97163767 2.98278691]
Random Forest mean cross validation score: 3.008763135130173
Random Forest RMSE: 3.0501160451975635, Random Forest R-squared: 0.09538436740461675
----------------------------------------------------------------------------------------------------
                           Feature  Importance
2                            value    0.084776
34        total_points_last_season    0.056760

Forwards:

In [8]:
FWD_data = df[(df['position'] == 'FWD') & (df['over_60_minutes'] == 1)]
FWD_points_target = FWD_data['points']
FWD_points_features = FWD_data[['team_market_value', 'opponent_market_value', 'value', 'was_home','points_last_game', 'total_points', 'mins_last_game',
                        'total_mins', 'mean_points_last_3', 'mean_mins_last_3', 'mean_points_last_5','mean_mins_last_5', 'mean_points_last_10', 
                        'mean_mins_last_10', 'team_points_last_game', 'total_team_points', 'mean_team_points_last_3', 'mean_team_points_last_5',
                        'mean_team_points_last_10', 'team_conceded_last_game', 'total_team_conceded', 
                        'mean_team_conceded_last_3', 'mean_team_conceded_last_5', 'mean_team_conceded_last_10', 'total_opponent_points',
                        'opponent_points_last_game', 'mean_opponent_points_last_3', 'mean_opponent_points_last_5', 'mean_opponent_points_last_10',
                        'total_opponent_conceded', 'opponent_conceded_last_game', 'mean_opponent_conceded_last_3', 'mean_opponent_conceded_last_5',
                        'mean_opponent_conceded_last_10', 'total_points_last_season', 'total_mins_last_season', 'total_team_points_last_season',
                        'total_team_conceded_last_season', 'total_opponent_points_last_season', 'total_opponent_conceded_last_season']]

x_train, x_test, y_train, y_test = train_test_split(FWD_points_features, FWD_points_target, train_size=0.8, test_size=0.2)

# Random Forest Regressor

best_FWD_rf_params = best_hyperparameters[('FWD', 'random_forest')]

FWD_rf_reg = RandomForestRegressor(n_estimators=best_FWD_rf_params['n_estimators'], min_samples_split=best_FWD_rf_params['min_samples_split'],
                                max_depth=best_FWD_rf_params['max_depth'], min_samples_leaf=best_FWD_rf_params['min_samples_leaf'],
                                max_features=best_FWD_rf_params['max_features'], random_state=42)

cv_scores = cross_val_score(GK_rf_reg, x_train, y_train, cv=5, scoring= 'neg_root_mean_squared_error', n_jobs=-1)
FWD_rf_reg.fit(x_train, y_train)
y_pred = FWD_rf_reg.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = FWD_rf_reg.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': FWD_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))

print(f'Random Forest cross validation scores: {np.abs(cv_scores)}')
print(f'Random Forest mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'Random Forest RMSE: {RMSE}, Random Forest R-squared: {R_squared}')
print('-'*100)

# Gradient Boosting Regressor

best_FWD_gb_params = best_hyperparameters[('FWD', 'gradient_boosting')]

FWD_gb_reg = GradientBoostingRegressor(n_estimators=best_FWD_gb_params['n_estimators'], learning_rate=best_FWD_gb_params['learning_rate'],
                                    max_depth=best_FWD_gb_params['max_depth'], max_features=best_FWD_gb_params['max_features'],
                                    min_samples_leaf=best_FWD_gb_params['min_samples_leaf'], min_samples_split=best_FWD_gb_params['min_samples_split'],
                                    subsample=best_FWD_gb_params['subsample'] )

cv_scores = cross_val_score(FWD_gb_reg, x_train, y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
FWD_gb_reg.fit(x_train, y_train)
y_pred = FWD_gb_reg.predict(x_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
R_squared = r2_score(y_test, y_pred)

importances = FWD_gb_reg.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': FWD_points_features.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(10))

print(f'Gradient Boosting cross validation scores: {np.abs(cv_scores)}')
print(f'Gradient Boosting mean cross validation score: {np.abs(np.mean(cv_scores))}')
print(f'Gradient Boosting RMSE: {RMSE}, Gradient Boosting R-squared: {R_squared}')

                         Feature  Importance
2                          value    0.106034
1          opponent_market_value    0.062942
17       mean_team_points_last_5    0.054457
29       total_opponent_conceded    0.033822
14         team_points_last_game    0.033608
25     opponent_points_last_game    0.031956
28  mean_opponent_points_last_10    0.029006
12           mean_points_last_10    0.028813
0              team_market_value    0.028452
30   opponent_conceded_last_game    0.027978
Random Forest cross validation scores: [3.43055778 3.57524155 3.438727   3.56584784 3.50465278]
Random Forest mean cross validation score: 3.5030053883569274
Random Forest RMSE: 3.7070688968975594, Random Forest R-squared: 0.017804071762034135
----------------------------------------------------------------------------------------------------
                         Feature  Importance
2                          value    0.042112
1          opponent_market_value    0.040435
17       mean_team_points